# 介紹
使用 pyautogui 來完成小恐龍的遊戲機器人。

小恐龍是在 chrome 中使用 chrome://dino 啟動。

邏輯：

持續監視仙人掌與鳥類會經過的掃描線。
每一幀以各掃描線上的主要 RGB 顏色估計背景，再找出與背景差異明顯且連續的像素。
白天、夜晚及轉場時皆使用當前畫面估算，不依賴固定障礙物顏色。

In [117]:
from collections import Counter
import importlib
import pyscreeze  # 若此處出錯，會顯示截圖套件的實際錯誤原因
import pyautogui
pyg = importlib.reload(pyautogui)  # 重新載入截圖功能
test_image = pyg.screenshot()
print('截圖測試成功，螢幕大小：', test_image.size)

# 在仙人掌會經過的掃描線檢查指定顏色；座標是螢幕座標。
x_start, x_end, y = 400, 470, 666
bird_y = 568
RGB_DIFFERENCE_THRESHOLD = 45
MIN_CONSECUTIVE_PIXELS = 1

def detect_line(image, line_y, background_color):
    pixels = [image.getpixel((x, line_y))[:3] for x in range(x_start, min(x_end, image.width))]
    if not pixels:
        return False, background_color
    current_mode = Counter(pixels).most_common(1)[0][0]
    if background_color is None:
        background_color = current_mode
    elif max(abs(current_mode[i] - background_color[i]) for i in range(3)) >= RGB_DIFFERENCE_THRESHOLD:
        changed = sum(max(abs(pixel[i] - background_color[i]) for i in range(3)) >= RGB_DIFFERENCE_THRESHOLD for pixel in pixels)
        if changed >= max(2, int(len(pixels) * 0.8)):
            background_color = current_mode
    run = 0
    detected = False
    for pixel in pixels:
        difference = max(abs(pixel[i] - background_color[i]) for i in range(3))
        if difference >= RGB_DIFFERENCE_THRESHOLD:
            run += 1
            if run >= MIN_CONSECUTIVE_PIXELS:
                detected = True
                break
        else:
            run = 0
    return detected, background_color

print('請在 5 秒內切換到 Chrome 恐龍遊戲視窗。')
pyg.sleep(5)
pyg.press('space')  # 開始遊戲
was_obstacle = False
was_cactus = False
was_bird = False
cactus_background = None
bird_background = None

try:
    while True:
        im = pyg.screenshot()
        cactus_detected, cactus_background = detect_line(im, y, cactus_background)
        bird_detected, bird_background = detect_line(im, bird_y, bird_background)
        obstacle_detected = cactus_detected or bird_detected
        new_cactus = cactus_detected and not was_cactus
        new_bird = bird_detected and not was_bird
        new_other_line = (new_cactus and was_bird) or (new_bird and was_cactus)

        # 同一個障礙物只觸發一次，離開掃描線後才重新待命。
        if obstacle_detected and (not was_obstacle or new_other_line):
            pyg.press('space')
        was_obstacle = obstacle_detected
        was_cactus = cactus_detected
        was_bird = bird_detected
except KeyboardInterrupt:
    print('已停止恐龍遊戲機器人。')

截圖測試成功，螢幕大小： (1920, 1080)
請在 5 秒內切換到 Chrome 恐龍遊戲視窗。
已停止恐龍遊戲機器人。


In [ ]:
# 簡單確認仙人掌與鳥類兩條線的偵測結果
im = pyg.screenshot()
cactus_found, _ = detect_line(im, y, cactus_background)
bird_found, _ = detect_line(im, bird_y, bird_background)
print(f'仙人掌 y={y}: {cactus_found}; 鳥類 y={bird_y}: {bird_found}')